In [7]:
import copy
import multiprocessing
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
from loguru import logger
import datetime
import pygmo as pg

from solhycool_modeling import EnvironmentVariables
from solhycool_optimization.problems.static import DcProblem as Problem
from solhycool_optimization.utils.evaluation import optimize

%load_ext autoreload
%autoreload 2

logger.disable("phd_visualizations")

data_path: Path = Path("../../data")
env_path: Path = data_path / "datasets/environment_data_20220101_20241231.h5"
base_output_path: Path = Path("../results")

# cc_model = combined_cooler_model.initialize()

def optimize_wrapper(problem) -> dict:
    operation_points, _, _, fitness_list = optimize(
        problem, extra_outputs=True, max_iter=300, use_mbh=True, 
        algo_id="compass_search", n_trials=2, log_verbosity=1,
        initial_pop_size=50
    )
    best_idx = np.argmin(fitness_list[:, 0])
    
    return asdict(operation_points[best_idx])

def process_entry(args) -> tuple[pd.DatetimeIndex, dict]:
    idx, dt, ds = args
    print(f"Iteration {idx+1} out of {len(df_)}: {dt}")
    env_vars = EnvironmentVariables.from_series(ds)
    env_vars.Q = env_vars.Q / 2
    env_vars.mv = env_vars.mv / 2
    
    problem = Problem(env_vars=env_vars, debug_mode=False)
    
    print(copy.deepcopy(problem))
    
    result = optimize_wrapper(problem)
    
    return dt, result




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
df_env = pd.read_hdf(env_path)

selected_date_str: str = "20220103" 
df_ = df_env.loc[selected_date_str]

env_vars = EnvironmentVariables.from_series(df_.iloc[0])
env_vars.Q = env_vars.Q / 2
env_vars.mv = env_vars.mv / 2

print("Before deepcopying")
problem = Problem(env_vars=env_vars, debug_mode=False)
print(
    f"works: {problem.fitness(pg.random_decision_vector(pg.problem(problem)))}"
)
problem = copy.deepcopy(problem)
print("After deepcopying")
problem.fitness(pg.random_decision_vector(pg.problem(problem)))


2025-03-04 09:06:32.043 | WARNING  | solhycool_optimization.problems.static:__init__:97 - Tamb=7.7 outside range: (9.06, 38.75), setting to: 9.06


Before deepcopying
I was called!
I was initialized!
works: [1.6798771536733135, 0.9633115153235288, -4.163736936436713, -9.083736936436715, 0.32373693643670975, -6.751954253539864]
After deepcopying
I was called!
I was initialized!


[3.458067391102064,
 0.6681089667221052,
 -4.0609269608725,
 -8.980926960872502,
 0.22092696087249664,
 -2.4621800761560024]

In [4]:
problem._cc_model


In [ ]:
df_env = pd.read_hdf(env_path)

selected_date_str: str = "20220103" 
df_ = df_env.loc[selected_date_str]

output_path = base_output_path / selected_date_str
if not output_path.exists():
    output_path.mkdir(parents=True)

results: list[dict] = []

# with multiprocessing.Pool() as pool:
#     results = pool.map(process_entry, [(idx, dt, ds) for idx, (dt, ds) in enumerate(df_.iterrows())])
results = [process_entry((idx, dt, ds)) for idx, (dt, ds) in enumerate(df_.iterrows())]

df_results = pd.DataFrame([r[1] for r in results], index=[r[0] for r in results])
df_results.to_csv(output_path / "dc_preliminary_results.csv")
logger.info(f"Results saved in {output_path}")
df_results


In [1]:
import pandas as pd

file_path = "/workspaces/SOLhycool/simulation/results/20220101_20220601/df_sim_eval_at_20250304T1234"

df = pd.read_hdf(f"{file_path}.h5")

display(df)

df.to_csv(f"{file_path}.csv")


,Tamb,HR,mv,Tv,qc,Tc_in,Tc_out,Tcond,Qc_released,Qc_absorbed,...,Vavail_s1,Je,Jw,J,Je_c,Je_dc,Je_wct,Jw_wct,Jw_s1,Jw_s2
2022-01-01 00:00:00+00:00,10.70,89.25,148.887083,35.0,5.456644,21.105831,36.894169,35.0,100.0,100.032597,...,19.083485,0.371712,0.0,0.371712,0.024592,0.347120,0.0,0.0,0.0,0.0
2022-01-01 01:00:00+00:00,11.60,82.00,148.887083,35.0,5.954455,21.765808,36.234192,35.0,100.0,100.032597,...,19.020231,0.622315,0.0,0.622315,0.026737,0.595578,0.0,0.0,0.0,0.0
2022-01-01 02:00:00+00:00,11.30,82.00,148.887083,35.0,5.763825,21.526548,36.473452,35.0,100.0,100.032597,...,19.002447,0.348633,0.0,0.348633,0.022138,0.326494,0.0,0.0,0.0,0.0
2022-01-01 03:00:00+00:00,11.20,82.00,148.887083,35.0,10.189383,24.772495,33.227505,35.0,100.0,100.032597,...,18.965136,0.102376,0.0,0.102376,0.057725,0.044651,0.0,0.0,0.0,0.0
2022-01-01 04:00:00+00:00,11.00,82.00,148.887083,35.0,6.138409,21.982600,36.017400,35.0,100.0,100.032597,...,18.968172,0.136408,0.0,0.136408,0.023314,0.113094,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-01-08 19:00:00+00:00,9.06,88.00,148.887083,35.0,6.297222,22.159575,35.840425,35.0,100.0,100.032597,...,18.935059,0.349138,0.0,0.349138,0.052816,0.296322,0.0,0.0,0.0,0.0
2022-01-08 20:00:00+00:00,9.06,86.00,148.887083,35.0,7.617986,23.345531,34.654469,35.0,100.0,100.032597,...,19.051190,0.252286,0.0,0.252286,0.070122,0.182165,0.0,0.0,0.0,0.0
2022-01-08 21:00:00+00:00,9.06,89.25,148.887083,35.0,6.467437,22.339608,35.660392,35.0,100.0,100.032597,...,18.944719,0.290536,0.0,0.290536,0.048279,0.242257,0.0,0.0,0.0,0.0
2022-01-08 22:00:00+00:00,9.06,88.00,148.887083,35.0,8.165864,23.724910,34.275090,35.0,100.0,100.032597,...,19.056404,0.168612,0.0,0.168612,0.056028,0.112584,0.0,0.0,0.0,0.0
